# Continuous Control

Every environment we have solved so far had a handful of discrete actions: left or right, four directions, four engines. Many of the most interesting applications of reinforcement learning are not like this. A robot arm is controlled by the torques applied to its joints, a car by the steering angle and the throttle, a chemical plant by valve openings and temperatures. In all these cases the action is a **vector of real numbers**, and the action space is a continuous set, typically a box $\mathcal{A} = [a_{\min}, a_{\max}]^{d}$.

This change breaks the value-based methods of Chapter 07 at their core. Q-learning, NFQ and DQN all need, at every step and in every target, a **maximum over actions**: $\max_{a} Q(s,a)$. With two actions this is a comparison; with infinitely many it is an optimisation problem in its own right, which we would have to solve numerically at every time step and for every transition in every mini-batch. Discretising the action space does not help much: a robot with seven joints and ten torque levels per joint has $10^7$ actions. The methods of Chapters 08 and 09, instead, do not break at all. A parameterised policy can output the parameters of a probability density over actions, a Gaussian for instance, and the policy gradient theorem, REINFORCE, A2C and GAE apply **without any change**: the only thing that changes is the form of $\log \pi_\theta(a \mid s)$. This is the first route to continuous control, and it is the one followed by PPO in Chapter 11.

This chapter follows a second route, which leads to algorithms that are far more **sample efficient**. The idea is to combine the best of the two families we know: the **actor-critic** structure of Chapter 09, with a policy network that outputs the action directly, and the **off-policy machinery** of Chapter 07, with a replay memory and target networks, so that every transition can be reused many times. The key that makes this combination possible is a different kind of policy gradient, the **deterministic policy gradient**, in which the critic does not just evaluate the action taken but tells the actor *in which direction to change it*. We develop the idea through three algorithms that build on one another: **DDPG**, which is essentially DQN with an actor; **TD3**, which fixes its most serious instabilities with three small changes; and **SAC**, which brings back a stochastic policy and adds entropy to the objective itself, and is today the standard off-policy algorithm for continuous control. We develop them on the Pendulum environment and conclude with the continuous version of Lunar Lander.

## The Pendulum Environment

The classic test bed for continuous control is the **inverted pendulum swing-up**. A pendulum hangs from a pivot and the agent applies a torque to the pivot; the goal is to swing the pendulum up and keep it balanced upright. The torque is limited, and it is not enough to lift the pendulum directly: the agent must first swing it back and forth to accumulate energy, then catch it at the top. The state has three components, the cosine and sine of the angle $\theta$ (measured from the upright position) and the angular velocity $\dot\theta$; the action is a single real number, the torque $u \in [-2, 2]$. The reward penalises the distance from the upright position, the angular velocity and the control effort,

$\displaystyle R_{t+1} = -\big( \theta_t^2 + 0.1\, \dot\theta_t^2 + 0.001\, u_t^2 \big)$

so that it is always non-positive, with a maximum of $0$ when the pendulum is upright, still and no torque is applied. Episodes have no terminal state and are truncated after $200$ steps, so the return of an episode is a sum of $200$ negative rewards: a random policy collects about $-1200$ per episode, while a good policy, which swings up in the first few dozen steps and then balances, collects between $-100$ and $-200$ depending on the initial position. Note that, since the episode never terminates, **every episode boundary is a truncation**: the agent must always bootstrap on the last state, a point that we will have to handle correctly.

In [ ]:
import gymnasium

pendulum = gymnasium.make('Pendulum-v1', render_mode="rgb_array")

print('State Space: ', pendulum.observation_space)
print('Action Space: ', pendulum.action_space)

The action space is a `Box`: it has a shape, here a single dimension, and lower and upper bounds. We read both from the environment, since the actor will have to produce actions within the bounds:

In [ ]:
import torch
import numpy as np
import random

# set the seeds for reproducibility of results
seed = 34

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

pendulum.reset(seed=seed)
pendulum.action_space.seed(seed)

state_size = pendulum.observation_space.shape[0]
action_size = pendulum.action_space.shape[0]
action_max = float(pendulum.action_space.high[0])

print('State size: ', state_size)
print('Action size: ', action_size)
print('Maximum action: ', action_max)

The evaluation and visualisation functions are those of the previous chapters. The only difference is that a policy now returns an array of real numbers rather than an integer; the random policy samples uniformly from the box:

In [ ]:
def evaluate(env, policy, episodes=1):

    # collect total rewards per episode
    rewards = []

    for episode in range(episodes):

        # reset the environment
        state, _ = env.reset()
        done = False
        total_reward = 0.0

        # run one episode
        while not done:
            action = policy(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

        rewards.append(total_reward)

    # return the average reward over episodes
    return float(np.mean(rewards))

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

def show_policy(env, policy, n_episodes=1, max_steps=200):

    for episode in range(n_episodes):

        # reset the environment
        state, _ = env.reset()
        done = False
        step = 0

        # run the episode until done or max_steps
        while not done and step < max_steps:

            # make a step
            action = policy(state)
            state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # render the environment
            clear_output(wait=True)
            plt.imshow(env.render())
            plt.axis("off")
            plt.title(f"Episode {episode+1}, Step {step}")
            plt.show()
            plt.close()

            step += 1

In [ ]:
def random_pi(state):
    # a torque sampled uniformly within the bounds
    return pendulum.action_space.sample()

random_score = evaluate(pendulum, random_pi, episodes=100)
print("Random score: ", random_score)

In [ ]:
show_policy(pendulum, random_pi, n_episodes=1, max_steps=100)

## Stochastic Policies for Continuous Actions

Before turning to the deterministic route, it is worth seeing how little the methods of Chapters 08 and 09 need to change for continuous actions, because the same ingredients will reappear in SAC. For a discrete action space the policy network produced one preference per action and the softmax turned them into probabilities. For a continuous action space the network produces the **parameters of a probability density**. The standard choice is a Gaussian with a state-dependent mean and standard deviation, independent across the $d$ components of the action:

$\displaystyle \pi_\theta(a \mid s) = \prod_{i=1}^{d} \mathcal{N}\big( a_i;\ \mu_{\theta,i}(s),\ \sigma_{\theta,i}(s)^2 \big)$

The network outputs the mean $\mu_\theta(s)$ directly and, to keep the standard deviation positive, the **logarithm** of $\sigma_\theta(s)$ (or a single state-independent $\log\sigma$ vector as an extra parameter). Everything else is unchanged. Sampling an action means drawing $a = \mu_\theta(s) + \sigma_\theta(s) \odot \xi$ with $\xi \sim \mathcal{N}(0, I)$. The log-density of the action taken, which replaces $\log \pi_\theta(a \mid s)$ in every formula of the previous chapters, is

$\displaystyle \log \pi_\theta(a \mid s) = \sum_{i=1}^{d} \left[ -\frac{(a_i - \mu_i)^2}{2\sigma_i^2} - \log \sigma_i - \frac{1}{2} \log 2\pi \right]$

and its gradient with respect to the mean and standard deviation, the **score function** of a Gaussian, has a transparent meaning:

$\displaystyle \nabla_{\mu_i} \log \pi_\theta(a \mid s) = \frac{a_i - \mu_i}{\sigma_i^2}, \qquad \nabla_{\sigma_i} \log \pi_\theta(a \mid s) = \frac{(a_i - \mu_i)^2 - \sigma_i^2}{\sigma_i^3}$

Weighted by a positive advantage, the first pushes the mean *towards* the action that was taken; the second increases $\sigma$ if the good action was further from the mean than one standard deviation and decreases it otherwise. The entropy, which replaces the entropy of the softmax in the bonus term, is $H = \sum_i \big( \log \sigma_i + \tfrac{1}{2} \log 2\pi e \big)$: the entropy bonus simply keeps the standard deviations from shrinking too fast. Finally, since a Gaussian has unbounded support while the action is bounded, the sampled action must be **clipped** to the box before being sent to the environment, or **squashed** with a $\tanh$, a choice we will examine in detail with SAC.

With these changes REINFORCE, VPG and A2C work as they are, and so does PPO, as we will see in Chapter 11. But they also keep their main limitation: they are **on-policy**, every transition is used once, and continuous control tasks tend to need millions of them. The rest of this chapter looks for a way to reuse experience.

## The Deterministic Policy Gradient

### From a Maximum to a Gradient

Let us go back to what we would like to do, namely Q-learning with a neural network, and to what stops us: the maximum over actions. Suppose we have a critic $Q_\phi(s,a)$ that takes the state **and the action** as input and returns a scalar; this is the first architecture we considered in Chapter 07 and discarded because it required one forward pass per action, a cost that is irrelevant now that there is no finite set of actions to enumerate. The greedy policy would be

$\displaystyle \mu(s) = \arg\max_{a} Q_\phi(s,a)$

which we cannot compute. But we can do something else: **learn** the maximiser. Let a second network, the **actor** $\mu_\theta(s)$, output an action for each state, and train it so that $Q_\phi(s, \mu_\theta(s))$ is as large as possible. The objective of the actor is therefore

$\displaystyle J(\theta) = \mathbb{E}_{s \sim \rho}\big[ Q_\phi\big(s, \mu_\theta(s)\big) \big]$

where $\rho$ is some distribution over states, and its gradient follows from the **chain rule**:

$\displaystyle \nabla_\theta J(\theta) = \mathbb{E}_{s \sim \rho}\Big[ \nabla_\theta \mu_\theta(s)\ \nabla_a Q_\phi(s,a)\big|_{a = \mu_\theta(s)} \Big]$

This chain rule is only part of the story, because we differentiated a *fixed* critic, whereas the performance we really care about is the expected return $J(\theta) = \mathbb{E}[\sum_t \gamma^t R_{t+1}]$ of the policy $\mu_\theta$, and the true action-value function $q_{\mu_\theta}$ depends on $\theta$ too. The [**deterministic policy gradient theorem**](./papers/2014%20-%20Deterministic%20Policy%20Gradient%20Algorithms.pdf) of Silver and colleagues (2014) states that this dependence contributes nothing, exactly as the state distribution contributed nothing in Chapter 08:

$\displaystyle \nabla_\theta J(\theta) = \mathbb{E}_{s \sim \rho^{\mu_\theta}}\Big[ \nabla_\theta \mu_\theta(s)\ \nabla_a q_{\mu_\theta}(s,a)\big|_{a = \mu_\theta(s)} \Big]$

where $\rho^{\mu_\theta}$ is the (discounted) distribution of the states visited under the policy. The implementation replaces the true $q_{\mu_\theta}$ with the critic $Q_\phi$, which is the formula above. The two factors have a clear meaning. The second, $\nabla_a Q_\phi(s,a)$, is the critic's opinion about **how the action should change** to obtain a higher value: it is a vector in action space, which exists only because actions are continuous. The first, $\nabla_\theta \mu_\theta(s)$, tells how to change the parameters of the actor to move its output in that direction. The actor follows the critic's gradient, state by state, climbing the landscape of $Q$ in action space. In practice we do not compute the two factors separately: the loss $-Q_\phi(s, \mu_\theta(s))$ is a differentiable function of $\theta$ through the actor's output, and backpropagation does the rest.

### Why It Allows Off-Policy Learning

Compare this with the stochastic policy gradient of Chapter 08. There, the gradient was an expectation over **actions sampled from the policy**, weighted by a score function: this is what forced us to use data from the current policy. Here, the actor is deterministic, there is nothing to sample, and the expectation is only over **states**. Which states? In the exact theorem, states should be distributed as they are under the current policy. In practice we take them from a **replay memory**, filled by past versions of the policy plus exploration noise, as in DQN: this is an approximation, since the state distribution does not match the one in the theorem, but it is the same approximation that every policy gradient implementation makes when it drops the discount factor $\gamma^t$, and it works well in practice. The result is that **the deterministic policy gradient can be estimated off-policy**, from stored transitions, with no importance weights. The critic, in turn, is trained exactly as in DQN, with one difference: the maximum over actions in the target is replaced by the actor's action. We have everything we need to reuse experience.

There is a price to pay, and it is exploration. A deterministic policy always takes the same action in the same state: it has no randomness of its own, so exploration must be added from outside, by perturbing the actions sent to the environment with noise. Since learning is off-policy, this is harmless for the update: the critic learns from whatever transitions it is given, and the actor follows the critic.

## Deep Deterministic Policy Gradient (DDPG)

[**DDPG**](./papers/2016%20-%20Continuous%20Control%20with%20Deep%20Reinforcement%20Learning.pdf), introduced by Lillicrap and colleagues in 2016, is the direct combination of the deterministic policy gradient with the two stabilising ingredients of DQN. It has **four networks**: the actor $\mu_\theta$, the critic $Q_\phi$, and a **target copy of each**, $\mu_{\theta^-}$ and $Q_{\phi^-}$. The critic is trained to minimise the mean squared TD error on mini-batches sampled uniformly from the replay memory $\mathcal{D}$, with a target computed entirely by the target networks:

$\displaystyle y = r + \gamma\, (1-d)\, Q_{\phi^-}\big( s', \mu_{\theta^-}(s') \big), \qquad \mathcal{L}(\phi) = \mathbb{E}_{(s,a,r,s',d) \sim U(\mathcal{D})} \Big[ \big( y - Q_\phi(s,a) \big)^2 \Big]$

This is the DQN loss with $\max_{a'} Q_{\phi^-}(s',a')$ replaced by $Q_{\phi^-}(s', \mu_{\theta^-}(s'))$: the target actor plays the role of the $\arg\max$. The actor is trained to maximise the critic's evaluation of its own actions on the same mini-batch:

$\displaystyle \mathcal{L}(\theta) = - \mathbb{E}_{s \sim U(\mathcal{D})}\big[ Q_\phi\big(s, \mu_\theta(s)\big) \big]$

The critic update is a semi-gradient update in the sense of Chapter 07: the target $y$ is treated as a constant. In the actor update the gradient flows through the critic, but only the actor's parameters $\theta$ are updated.

### Soft Target Updates

DQN copied the online weights into the target network every $C$ steps (a **hard** update). DDPG uses **soft** updates instead: after every optimization step the target parameters are moved a small fraction of the way towards the online parameters,

$\displaystyle \theta^- \leftarrow \tau\, \theta + (1-\tau)\, \theta^-, \qquad \phi^- \leftarrow \tau\, \phi + (1-\tau)\, \phi^-$

with $\tau$ small (we use $0.005$). The target networks then track the online networks with an exponential delay, with time constant $1/\tau$ steps, instead of jumping every $C$ steps: the targets change a little at every step rather than a lot once in a while. With two networks that must stay consistent with each other, this smoother behaviour is important. (This update is sometimes called a *Polyak* update; it was already mentioned in the exercises of Chapter 07.)

### Exploration Noise

As discussed, the deterministic actor does not explore. During data collection we add noise to its action and clip the result to the action bounds:

$\displaystyle A_t = \text{clip}\big( \mu_\theta(S_t) + \epsilon_t,\ a_{\min},\ a_{\max} \big), \qquad \epsilon_t \sim \mathcal{N}(0, \sigma^2 I)$

The original paper used temporally correlated noise (an Ornstein-Uhlenbeck process, in which successive perturbations are similar, so that the agent explores by drifting rather than jittering); later work showed that independent Gaussian noise works just as well, and we use it. The standard deviation $\sigma$ is expressed as a fraction of the action bound (we use $0.1\, a_{\max}$). In addition, as in DQN, we fill the replay memory with a number of **random transitions** before training starts, so that the first mini-batches are diverse.

### Networks

The actor maps the state to an action. Its output layer has one unit per action dimension, followed by a $\tanh$, which produces values in $(-1, 1)$, scaled by the maximum action. Since the network needs this final scaling, we write it as a small PyTorch module rather than a plain `Sequential`:

In [ ]:
class Actor(torch.nn.Module):

    def __init__(self, env, first_hidden_layer=256, second_hidden_layer=256):
        super().__init__()

        state_size = env.observation_space.shape[0]
        action_size = env.action_space.shape[0]

        # the bound of the action box, used to scale the output
        self.action_max = float(env.action_space.high[0])

        self.network = torch.nn.Sequential(
            torch.nn.Linear(state_size, first_hidden_layer),
            torch.nn.ReLU(),
            torch.nn.Linear(first_hidden_layer, second_hidden_layer),
            torch.nn.ReLU(),
            torch.nn.Linear(second_hidden_layer, action_size),
            # squash the output into (-1, 1) ...
            torch.nn.Tanh(),
        )

    def forward(self, state):
        # ... and scale it to the action bounds
        return self.action_max * self.network(state)

The critic receives the state **and** the action, concatenated into a single input vector, and outputs one scalar, the estimated action value:

In [ ]:
class Critic(torch.nn.Module):

    def __init__(self, env, first_hidden_layer=256, second_hidden_layer=256):
        super().__init__()

        state_size = env.observation_space.shape[0]
        action_size = env.action_space.shape[0]

        self.network = torch.nn.Sequential(
            # the input is the state concatenated with the action
            torch.nn.Linear(state_size + action_size, first_hidden_layer),
            torch.nn.ReLU(),
            torch.nn.Linear(first_hidden_layer, second_hidden_layer),
            torch.nn.ReLU(),
            # output: the value Q(s, a)
            torch.nn.Linear(second_hidden_layer, 1),
        )

    def forward(self, state, action):
        return self.network(torch.cat([state, action], dim=-1)).squeeze(-1)

We create the four networks, initialising each target as an exact copy of its online network, and the two optimizers. Following common practice, the critic uses a larger learning rate than the actor, for the same reason discussed in Chapter 09: the critic must keep up with a moving policy, and the actor must move slowly enough for the critic's gradient to remain meaningful:

In [ ]:
import copy

actor = Actor(pendulum)
critic = Critic(pendulum)

target_actor = copy.deepcopy(actor)
target_critic = copy.deepcopy(critic)

actor_learning_rate = 0.0003
critic_learning_rate = 0.001

actor_optimizer = torch.optim.Adam(actor.parameters(), lr=actor_learning_rate)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=critic_learning_rate)

The soft update moves every parameter of a target network a fraction $\tau$ of the way towards the corresponding online parameter:

In [ ]:
def soft_update(target, online, tau=0.005):

    with torch.no_grad():
        for target_parameter, online_parameter in zip(target.parameters(), online.parameters()):
            target_parameter.mul_(1.0 - tau).add_(tau * online_parameter)

The deterministic policy, used for evaluation, is the actor's output; the exploratory policy, used to collect experience, adds Gaussian noise and clips to the bounds:

In [ ]:
def actor_pi(state):

    # the deterministic action of the actor
    with torch.no_grad():
        action = actor(torch.as_tensor(state, dtype=torch.float32))

    return action.numpy()


def noisy_pi(state, noise_std=0.1):

    # the actor's action, perturbed with Gaussian noise and clipped to the bounds
    action = actor_pi(state) + np.random.normal(0.0, noise_std * action_max, size=action_size)

    return np.clip(action, -action_max, action_max).astype(np.float32)

### Replay Memory

The replay memory is the circular buffer of Chapter 07, with one difference: the action is now a vector of floats rather than an integer. As in Chapter 07, the terminal flag records only genuine terminations; on Pendulum it is therefore always zero, since episodes are only truncated, and the target always bootstraps on the next state:

In [ ]:
# define the structured dtype for an experience tuple
experience_type = np.dtype([
    ('state',      np.float32, state_size),   # current state
    ('action',     np.float32, action_size),  # action taken (a vector)
    ('reward',     np.float32),               # reward received
    ('next_state', np.float32, state_size),   # next state
    ('failure',    np.int8)                   # 1 if the episode terminated (NOT truncated)
])

memory_size = 100_000

replay_memory = {
    "size": memory_size,
    "buffer": np.empty(memory_size, dtype=experience_type),
    "index": 0,
    "entries": 0,
}


def store_experience(experience):

    # store the transition at the current position
    replay_memory["buffer"][replay_memory["index"]] = experience

    # update the number of stored transitions and move to the next position
    replay_memory["entries"] = min(replay_memory["entries"] + 1, replay_memory["size"])
    replay_memory["index"] = (replay_memory["index"] + 1) % replay_memory["size"]


def sample_experiences(batch_size=256):

    # sample transitions uniformly at random and convert the fields to tensors
    indices = np.random.randint(0, replay_memory["entries"], size=batch_size)
    batch = replay_memory["buffer"][indices]

    return (
        torch.from_numpy(batch["state"].copy()).float(),
        torch.from_numpy(batch["action"].copy()).float(),
        torch.from_numpy(batch["reward"].copy()).float(),
        torch.from_numpy(batch["next_state"].copy()).float(),
        torch.from_numpy(batch["failure"].copy()).float(),
    )

### Loss Functions

One optimization step samples a mini-batch, updates the critic towards the target computed by the target networks, updates the actor along the critic's gradient, and softly updates both targets:

In [ ]:
def optimize_ddpg(gamma=0.99, batch_size=256, tau=0.005):

    states, actions, rewards, next_states, failures = sample_experiences(batch_size)

    # critic target, computed by the target networks: a constant
    with torch.no_grad():
        next_actions = target_actor(next_states)
        target = rewards + gamma * (1.0 - failures) * target_critic(next_states, next_actions)

    # critic loss: mean squared TD error
    critic_loss = (target - critic(states, actions)).pow(2).mean()

    critic_optimizer.zero_grad()
    critic_loss.backward()
    critic_optimizer.step()

    # actor loss: minus the critic's evaluation of the actor's actions
    # (the gradient flows through the critic, but only the actor is updated)
    actor_loss = -critic(states, actor(states)).mean()

    actor_optimizer.zero_grad()
    actor_loss.backward()
    actor_optimizer.step()

    # soft updates of the target networks
    soft_update(target_actor, actor, tau)
    soft_update(target_critic, critic, tau)

    return critic_loss.item(), actor_loss.item()

### Algorithm

The DDPG algorithm is the following:

1. Use four networks: the actor $\mu_\theta$, the critic $Q_\phi$, and a target copy of each, initialised as exact copies
2. Fill the replay memory with transitions generated by random actions
3. Act in the environment with the actor's action plus Gaussian exploration noise, clipped to the bounds, and store the transition
4. Sample a mini-batch from the replay memory; compute the target $y = r + \gamma(1-d)\, Q_{\phi^-}(s', \mu_{\theta^-}(s'))$ and update the critic on the squared TD error
5. Update the actor by gradient descent on $-Q_\phi(s, \mu_\theta(s))$
6. Softly update both target networks
7. Repeat from step 3 until the budget of environment steps is exhausted

As in Chapter 09, training is organised around environment steps, with one optimization step per environment step, and the deterministic policy is evaluated at regular intervals on a separate copy of the environment:

In [ ]:
def ddpg(env, max_steps, eval_every=1000, start_steps=1000, noise_std=0.1, gamma=0.99, batch_size=256, tau=0.005):

    # a separate, seeded environment for evaluation
    eval_env = gymnasium.make(env.spec.id)
    eval_env.reset(seed=int(env.np_random.integers(1_000_000)))

    scores = []
    state, _ = env.reset()

    for step in range(1, max_steps + 1):

        # random actions at the beginning, then the noisy actor
        if step <= start_steps:
            action = env.action_space.sample()
        else:
            action = noisy_pi(state, noise_std)

        # interact with the environment and store the transition
        next_state, reward, terminated, truncated, _ = env.step(action)
        store_experience((state, action, reward, next_state, terminated))

        # move on; reset the environment at the end of the episode
        state = next_state
        if terminated or truncated:
            state, _ = env.reset()

        # one optimization step per environment step, once the memory has enough transitions
        if step > start_steps:
            optimize_ddpg(gamma, batch_size, tau)

        # evaluate the deterministic policy at regular intervals
        if step % eval_every == 0:
            score = evaluate(eval_env, actor_pi, episodes=5)
            scores.append(score)
            print(f"Step {step:06}, score {score:07.1f}", end="\r", flush=True)

    return scores

### Solve Pendulum

We train the agent for a budget of environment steps. Pendulum is a small problem, and off-policy learning is sample efficient: a few tens of thousands of steps, that is a few hundred episodes, should be enough:

In [ ]:
scores = ddpg(pendulum, max_steps=30_000)

In [ ]:
agent_score = evaluate(pendulum, actor_pi, episodes=10)

print("Random score", random_score)
print("Agent score", agent_score)

In [ ]:
show_policy(pendulum, actor_pi, n_episodes=1, max_steps=200)

For the controlled experiments we reuse the step-based experiment and plotting functions of Chapter 09:

In [ ]:
import joblib
from pathlib import Path

def experiment(
    env,
    train,
    setup,
    max_steps,
    name,
    eval_every=1000,
    seeds=(12, 34, 56, 78, 90),
    smoothing_window=3,
):

    results_path = Path("temp") / f"{name}_results.joblib"

    # load saved results if they already exist
    if results_path.exists():
        experiment_results = joblib.load(results_path)

        n_points = len(experiment_results["mean_score"])
        final_mean = experiment_results["mean_score"][-1]
        final_min = experiment_results["min_score"][-1]
        final_max = experiment_results["max_score"][-1]

        print(f"Loaded saved results from {results_path}")
        print(f"Smoothed points: {n_points}")
        print(f"Smoothing window: {experiment_results['smoothing_window']}")
        print(f"Final score: mean={final_mean:.1f}, range=[{final_min:.1f}, {final_max:.1f}]")

        return experiment_results

    results = []

    for seed in seeds:
        print(f"Experiment seed: {seed}")

        # set seeds for reproducibility
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        env.reset(seed=seed)
        env.action_space.seed(seed)

        # initialize networks, optimizers, replay memory
        setup()

        # train the agent
        scores = train(env, max_steps=max_steps, eval_every=eval_every)

        # smooth the learning curve
        kernel = np.ones(smoothing_window) / smoothing_window
        scores = np.convolve(scores, kernel, mode="valid")

        results.append(scores)
        print()

    results = np.array(results)

    experiment_results = {
        "max_score": np.max(results, axis=0),
        "min_score": np.min(results, axis=0),
        "mean_score": np.mean(results, axis=0),
        "smoothing_window": smoothing_window,
        "eval_every": eval_every,
        "seeds": seeds,
        "max_steps": max_steps,
    }

    results_path.parent.mkdir(exist_ok=True)
    joblib.dump(experiment_results, results_path)

    return experiment_results

In [ ]:
def steps_axis(results):
    # the smoothed curves start after the first smoothing window
    offset = results.get("smoothing_window", 1)
    return (np.arange(len(results["mean_score"])) + offset) * results.get("eval_every", 1)


def plot_scores(results, title="Score", color="orange"):

    steps = steps_axis(results)

    plt.figure(figsize=(12, 6))
    plt.title(title)
    plt.xlabel("Environment steps")
    plt.ylabel("Score")

    plt.plot(steps, results["mean_score"], color=color, linewidth=2, label="Mean over seeds")
    plt.fill_between(steps, results["min_score"], results["max_score"], color=color, alpha=0.3, label="Range over seeds")

    plt.legend(loc="lower right")
    plt.show()


def plot_comparison(
    first_results,
    second_results,
    first_label="A",
    second_label="B",
    first_color="blue",
    second_color="red",
):

    first_steps = steps_axis(first_results)
    second_steps = steps_axis(second_results)

    plt.figure(figsize=(12, 6))
    plt.title(f"{second_label} vs {first_label}")
    plt.xlabel("Environment steps")
    plt.ylabel("Score")

    # plot the first algorithm
    plt.plot(first_steps, first_results["mean_score"], color=first_color, linewidth=2, label=f"{first_label} mean")
    plt.fill_between(first_steps, first_results["min_score"], first_results["max_score"], color=first_color, alpha=0.20, label=f"{first_label} range")

    # plot the second algorithm
    plt.plot(second_steps, second_results["mean_score"], color=second_color, linewidth=2, label=f"{second_label} mean")
    plt.fill_between(second_steps, second_results["min_score"], second_results["max_score"], color=second_color, alpha=0.20, label=f"{second_label} range")

    plt.legend(loc="lower right")
    plt.show()

The setup function creates fresh networks, optimizers and replay memory for each seed:

In [ ]:
# define the experiment setup and hyperparameters
gamma = 0.99                   # discount factor
tau = 0.005                    # soft update factor
batch_size = 256               # mini-batch size
memory_size = 100_000          # replay memory size
start_steps = 1000             # random transitions before training
noise_std = 0.1                # exploration noise (fraction of the action bound)
actor_learning_rate = 0.0003   # step size of the actor's optimizer
critic_learning_rate = 0.001   # step size of the critic's optimizer
first_hidden_layer = 256       # size of the first hidden layer
second_hidden_layer = 256      # size of the second hidden layer

def setup_ddpg():
    global actor, critic, target_actor, target_critic, actor_optimizer, critic_optimizer, replay_memory

    actor = Actor(pendulum, first_hidden_layer, second_hidden_layer)
    critic = Critic(pendulum, first_hidden_layer, second_hidden_layer)
    target_actor = copy.deepcopy(actor)
    target_critic = copy.deepcopy(critic)

    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=actor_learning_rate)
    critic_optimizer = torch.optim.Adam(critic.parameters(), lr=critic_learning_rate)

    replay_memory = {
        "size": memory_size,
        "buffer": np.empty(memory_size, dtype=experience_type),
        "index": 0,
        "entries": 0,
    }

def train_ddpg(env, max_steps, eval_every):
    return ddpg(env, max_steps, eval_every, start_steps, noise_std, gamma, batch_size, tau)

# run the experiment
ddpg_results = experiment(
    env=pendulum,
    train=train_ddpg,
    setup=setup_ddpg,
    max_steps=30_000,
    name="ddpg",
)

In [ ]:
plot_scores(ddpg_results, title="DDPG score", color="orange")

DDPG solves the swing-up: within a few thousand steps after the random phase the score jumps from the $-1200$ of the random policy to the range of a good controller, between $-100$ and $-200$, and the learned actor swings the pendulum up and balances it. Compared with the on-policy methods of the previous chapters this is remarkably **sample efficient**: a few thousand environment steps, that is a few dozen episodes, where REINFORCE needed hundreds of episodes on the much simpler Cart-Pole. The same transitions are replayed hundreds of times, and the critic's gradient gives the actor a direction to move in rather than a noisy score. The fluctuations of the curve after convergence are mostly **evaluation noise**: each evaluation averages five episodes with random initial angles, and an episode that starts with the pendulum hanging down costs more than one that starts near the top. On this small problem DDPG is reliable. On harder tasks, with higher-dimensional actions and less regular rewards, it is known to be **brittle**: sensitive to the learning rates, to the noise level, and above all to errors in the critic, which the actor exploits greedily. The next algorithm addresses exactly this.

## Twin Delayed DDPG (TD3)

The instability of DDPG has a precise origin. The actor is trained to **maximise the critic**, and the critic is a function approximator with errors. Wherever the critic overestimates the value of some action, the actor will move towards that action, because that is what maximising the critic means; the critic is then trained on the actions the actor proposes, with targets computed from its own overestimates, and the error can grow instead of being corrected. This is the **maximization bias** of Chapter 05 and Chapter 07 in a new form: there the maximum was taken explicitly over a finite set of noisy estimates, here it is performed by the actor, through gradient ascent on a noisy function, and the effect is the same. [**TD3**](./papers/2018%20-%20Addressing%20Function%20Approximation%20Error%20in%20Actor-Critic%20Methods.pdf), introduced by Fujimoto and colleagues in 2018, keeps the structure of DDPG and adds three modifications, each of them small and each aimed at a specific source of error.

### Clipped Double Q-Learning

Double DQN reduced the overestimation by separating action selection from action evaluation. TD3 takes a more direct approach: it trains **two critics**, $Q_{\phi_1}$ and $Q_{\phi_2}$, with different random initialisations, on the same data, and uses the **smaller** of their two estimates in the target:

$\displaystyle y = r + \gamma\, (1-d) \min_{i=1,2} Q_{\phi_i^-}\big( s', \tilde a \big)$

Both critics regress towards this common target. Taking the minimum of two noisy estimates biases the target **downwards**: a pessimistic estimate, which may slow learning a little but is far less harmful than an optimistic one, since the actor cannot exploit a value that is not there. The two critics agree where the data support the estimate and disagree where they are extrapolating, so the minimum penalises precisely the actions whose value is uncertain.

### Target Policy Smoothing

The second modification addresses the variance of the target. A deterministic target action $\mu_{\theta^-}(s')$ evaluated by the critic is sensitive to narrow peaks in the critic's estimate: if $Q$ is accidentally high at one specific action, the target picks it up exactly. TD3 **smooths** the target by adding clipped noise to the target action,

$\displaystyle \tilde a = \text{clip}\big( \mu_{\theta^-}(s') + \text{clip}(\epsilon, -c, c),\ a_{\min},\ a_{\max} \big), \qquad \epsilon \sim \mathcal{N}(0, \tilde\sigma^2 I)$

so that the target is effectively an average of $Q$ over a small neighbourhood of the target action. This is a form of regularisation: similar actions should have similar values, and the target enforces it. We use $\tilde\sigma = 0.2\, a_{\max}$ and $c = 0.5\, a_{\max}$, the values of the paper.

### Delayed Policy Updates

The third modification addresses the coupling between actor and critic. If the actor is updated on the basis of a critic that has not yet absorbed the effect of the previous actor update, the two chase each other. TD3 updates the actor, and the target networks, only **once every $k$ critic updates** (we use $k = 2$): the critic gets time to settle on a more accurate estimate before the actor moves again. This is the same idea as giving the critic a larger learning rate, pushed one step further.

### Algorithm

The TD3 algorithm is therefore DDPG with the following changes:

1. Use two critics, with their target copies; the actor and its target are as in DDPG
2. Compute the target action by adding clipped Gaussian noise to the target actor's action, and clip the result to the action bounds
3. Compute the target with the **minimum** of the two target critics, and update both critics on the squared TD error
4. Every $k$ critic updates, update the actor by gradient descent on $-Q_{\phi_1}(s, \mu_\theta(s))$ and softly update all the target networks

The actor update uses the first critic alone, as in the paper; using the minimum of the two is an equally common choice. We create the additional networks:

In [ ]:
actor = Actor(pendulum)
critic_1 = Critic(pendulum)
critic_2 = Critic(pendulum)

target_actor = copy.deepcopy(actor)
target_critic_1 = copy.deepcopy(critic_1)
target_critic_2 = copy.deepcopy(critic_2)

actor_optimizer = torch.optim.Adam(actor.parameters(), lr=actor_learning_rate)
critic_optimizer = torch.optim.Adam(list(critic_1.parameters()) + list(critic_2.parameters()), lr=critic_learning_rate)

The optimization step implements the three modifications. It receives the index of the current update, so that it can decide whether the actor is to be updated:

In [ ]:
def optimize_td3(update, gamma=0.99, batch_size=256, tau=0.005, target_noise_std=0.2, target_noise_clip=0.5, policy_delay=2):

    states, actions, rewards, next_states, failures = sample_experiences(batch_size)

    with torch.no_grad():

        # target policy smoothing: clipped noise on the target action
        noise = (torch.randn_like(actions) * target_noise_std * action_max).clamp(-target_noise_clip * action_max, target_noise_clip * action_max)
        next_actions = (target_actor(next_states) + noise).clamp(-action_max, action_max)

        # clipped double Q-learning: the minimum of the two target critics
        next_q = torch.min(target_critic_1(next_states, next_actions), target_critic_2(next_states, next_actions))
        target = rewards + gamma * (1.0 - failures) * next_q

    # both critics regress towards the same target
    critic_loss = (target - critic_1(states, actions)).pow(2).mean() + (target - critic_2(states, actions)).pow(2).mean()

    critic_optimizer.zero_grad()
    critic_loss.backward()
    critic_optimizer.step()

    # delayed policy update: the actor and the targets move once every policy_delay updates
    if update % policy_delay == 0:

        actor_loss = -critic_1(states, actor(states)).mean()

        actor_optimizer.zero_grad()
        actor_loss.backward()
        actor_optimizer.step()

        soft_update(target_actor, actor, tau)
        soft_update(target_critic_1, critic_1, tau)
        soft_update(target_critic_2, critic_2, tau)

    return critic_loss.item()

The training loop is that of DDPG, with a counter of the optimization steps:

In [ ]:
def td3(env, max_steps, eval_every=1000, start_steps=1000, noise_std=0.1, gamma=0.99, batch_size=256, tau=0.005):

    # a separate, seeded environment for evaluation
    eval_env = gymnasium.make(env.spec.id)
    eval_env.reset(seed=int(env.np_random.integers(1_000_000)))

    scores = []
    update = 0
    state, _ = env.reset()

    for step in range(1, max_steps + 1):

        # random actions at the beginning, then the noisy actor
        if step <= start_steps:
            action = env.action_space.sample()
        else:
            action = noisy_pi(state, noise_std)

        # interact with the environment and store the transition
        next_state, reward, terminated, truncated, _ = env.step(action)
        store_experience((state, action, reward, next_state, terminated))

        # move on; reset the environment at the end of the episode
        state = next_state
        if terminated or truncated:
            state, _ = env.reset()

        # one optimization step per environment step, once the memory has enough transitions
        if step > start_steps:
            update += 1
            optimize_td3(update, gamma, batch_size, tau)

        # evaluate the deterministic policy at regular intervals
        if step % eval_every == 0:
            score = evaluate(eval_env, actor_pi, episodes=5)
            scores.append(score)
            print(f"Step {step:06}, score {score:07.1f}", end="\r", flush=True)

    return scores

### Solve Pendulum

We run the same experiment as for DDPG and compare the two:

In [ ]:
def setup_td3():
    global actor, critic_1, critic_2, target_actor, target_critic_1, target_critic_2, actor_optimizer, critic_optimizer, replay_memory

    actor = Actor(pendulum, first_hidden_layer, second_hidden_layer)
    critic_1 = Critic(pendulum, first_hidden_layer, second_hidden_layer)
    critic_2 = Critic(pendulum, first_hidden_layer, second_hidden_layer)
    target_actor = copy.deepcopy(actor)
    target_critic_1 = copy.deepcopy(critic_1)
    target_critic_2 = copy.deepcopy(critic_2)

    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=actor_learning_rate)
    critic_optimizer = torch.optim.Adam(list(critic_1.parameters()) + list(critic_2.parameters()), lr=critic_learning_rate)

    replay_memory = {
        "size": memory_size,
        "buffer": np.empty(memory_size, dtype=experience_type),
        "index": 0,
        "entries": 0,
    }

def train_td3(env, max_steps, eval_every):
    return td3(env, max_steps, eval_every, start_steps, noise_std, gamma, batch_size, tau)

# run the experiment
td3_results = experiment(
    env=pendulum,
    train=train_td3,
    setup=setup_td3,
    max_steps=30_000,
    name="td3",
)

In [ ]:
plot_comparison(
    ddpg_results,
    td3_results,
    first_label="DDPG",
    second_label="TD3",
    first_color="orange",
    second_color="blue",
)

On Pendulum both algorithms reach the same final performance. TD3 is, if anything, slightly **slower** at the beginning: the pessimistic targets and the delayed actor updates are designed to prevent the actor from running ahead of the critic, and on a problem where the critic is accurate enough for DDPG to work, they cost a little time without buying anything visible. The three modifications pay off on the harder continuous control benchmarks on which TD3 was introduced, where the critic's errors are larger: there the differences are large, and DDPG often fails entirely. Since TD3 adds only a second critic and a few lines of code, it has replaced DDPG as the standard deterministic actor-critic.

## Soft Actor-Critic (SAC)

DDPG and TD3 have a deterministic actor and explore by adding noise from outside, with a fixed standard deviation that has to be chosen by hand and does not adapt to the state: the same noise is used when the pendulum is hanging still, where large exploratory torques are useful, and when it is balanced upright, where they are destructive. [**Soft Actor-Critic**](./papers/2018%20-%20Soft%20Actor-Critic%20-%20Off-Policy%20Maximum%20Entropy%20Deep%20Reinforcement%20Learning%20with%20a%20Stochastic%20Actor.pdf), introduced by Haarnoja and colleagues in 2018, brings back a **stochastic** actor, while keeping everything that makes TD3 work (replay memory, twin critics, soft target updates), and makes exploration part of the objective itself.

### The Maximum Entropy Objective

In the previous chapters the entropy of the policy appeared as a regularisation term added to the loss, with a coefficient $\beta$. SAC puts the entropy **inside the return**. The agent maximises, at every time step, the reward *plus* the entropy of its policy in the current state, weighted by a **temperature** $\alpha$:

$\displaystyle J(\pi) = \mathbb{E}_\pi \left[ \sum_{t=0}^{\infty} \gamma^{t} \Big( R_{t+1} + \alpha\, H\big( \pi(\cdot \mid S_t) \big) \Big) \right]$

The difference with an entropy bonus is subtle but important: the entropy of *future* states now counts in the value of the *current* action. An action that leads to states where many actions are good, and the policy can stay random, is worth more than one leading to states where only one action works. The agent is rewarded for keeping its options open, and explores more where it matters. (The letter $\alpha$ is the one used in the literature; in this chapter it is not a step size, as it was in the tabular chapters.)

The value functions are modified accordingly. The **soft action value** of a state-action pair is the expected sum of future rewards and entropies, and the soft state value is the expected soft action value under the policy plus the entropy of the current state:

$\displaystyle V(s) = \mathbb{E}_{a \sim \pi}\big[ Q(s,a) - \alpha \log \pi(a \mid s) \big] = \mathbb{E}_{a \sim \pi}\big[ Q(s,a) \big] + \alpha\, H\big( \pi(\cdot \mid s) \big)$

since $H(\pi(\cdot \mid s)) = -\mathbb{E}_{a \sim \pi}[\log \pi(a \mid s)]$. The soft Bellman equation then reads $Q(s,a) = \mathbb{E}\big[ R + \gamma\, V(S') \big]$, and the critic's target becomes

$\displaystyle y = r + \gamma\, (1-d) \Big( \min_{i=1,2} Q_{\phi_i^-}(s', a') - \alpha \log \pi_\theta(a' \mid s') \Big), \qquad a' \sim \pi_\theta(\cdot \mid s')$

This is the TD3 target with two changes: the next action is **sampled from the current actor** rather than computed by a target actor (SAC has no target actor: the stochasticity of the policy plays the smoothing role of TD3's target noise), and the entropy term $-\alpha \log \pi$ is added, so that the critic learns the soft values.

### The Actor and the Reparameterisation Trick

The actor should choose, in each state, the distribution that maximises the soft value $\mathbb{E}_{a \sim \pi}[Q(s,a) - \alpha \log \pi(a \mid s)]$: the first term pulls towards actions with high value, the second towards spreading the probability mass. Its loss is therefore

$\displaystyle \mathcal{L}(\theta) = \mathbb{E}_{s \sim U(\mathcal{D})}\ \mathbb{E}_{a \sim \pi_\theta(\cdot \mid s)} \Big[ \alpha \log \pi_\theta(a \mid s) - \min_{i=1,2} Q_{\phi_i}(s,a) \Big]$

Here we meet a technical point that is worth understanding, because it distinguishes SAC from everything in Chapters 08 and 09. The expectation is over actions sampled from the policy we are differentiating. In the stochastic policy gradient we handled this with the log-derivative trick, obtaining an estimator of the form *score times value*, which is unbiased but has high variance. SAC uses the **reparameterisation trick** instead. Write the sampled action as a deterministic function of the state and of an independent noise variable:

$\displaystyle a = f_\theta(s, \xi) = \tanh\big( \mu_\theta(s) + \sigma_\theta(s) \odot \xi \big)\, a_{\max}, \qquad \xi \sim \mathcal{N}(0, I)$

(here $\mu_\theta(s)$ is the **mean** of the Gaussian, not a deterministic action as in DDPG and TD3). Now the randomness is in $\xi$, which does not depend on $\theta$, and the expectation can be rewritten over $\xi$:

$\displaystyle \nabla_\theta\, \mathbb{E}_{a \sim \pi_\theta}\big[ Q(s,a) \big] = \mathbb{E}_{\xi}\Big[ \nabla_\theta f_\theta(s,\xi)\ \nabla_a Q(s,a)\big|_{a = f_\theta(s,\xi)} \Big]$

This is the deterministic policy gradient again, averaged over the noise: the critic's gradient with respect to the action tells the actor in which direction to move the sampled action, and backpropagation through $f_\theta$ moves the mean and the standard deviation accordingly. The estimator has much lower variance than the score-function one, because it uses the *gradient* of $Q$ rather than its *value*; it requires a differentiable critic that takes the action as input, which is exactly what we have. The $\log \pi_\theta(a \mid s)$ term of the loss is differentiated in the same way, since the action is a function of $\theta$.

### The Squashed Gaussian

The actor outputs the mean $\mu_\theta(s)$ and the log-standard-deviation $\log \sigma_\theta(s)$ of a Gaussian over an unbounded variable $u$, and the action is $a = \tanh(u)\, a_{\max}$, which lies strictly inside the bounds. The $\tanh$ changes the density: by the change-of-variables formula, with $a_i = a_{\max} \tanh(u_i)$ and $\mathrm{d}a_i / \mathrm{d}u_i = a_{\max}(1 - \tanh^2 u_i)$,

$\displaystyle \log \pi_\theta(a \mid s) = \log \mathcal{N}\big( u;\ \mu_\theta(s), \sigma_\theta(s)^2 \big) - \sum_{i=1}^{d} \log\big( a_{\max}\, (1 - \tanh^2 u_i) \big)$

The correction term is large where the $\tanh$ saturates: an action close to the bounds has a much higher density than the underlying Gaussian value, because a wide range of $u$ maps onto a narrow range of $a$. Omitting the correction would make the entropy term wrong exactly at the bounds, where the agent most often ends up (full torque).

### Automatic Temperature

The temperature $\alpha$ trades reward against entropy, and its right value depends on the scale of the rewards, which changes from environment to environment and even during training. Rather than tuning it by hand, SAC adjusts it automatically so that the policy keeps a **target entropy** $\bar H$: the temperature is increased when the entropy falls below the target and decreased when it exceeds it. This is obtained by gradient descent on

$\displaystyle \mathcal{L}(\alpha) = \mathbb{E}_{a \sim \pi_\theta}\big[ -\alpha \big( \log \pi_\theta(a \mid s) + \bar H \big) \big]$

whose gradient with respect to $\alpha$ is $-\mathbb{E}[\log \pi_\theta(a \mid s) + \bar H] = H(\pi_\theta(\cdot \mid s)) - \bar H$ with the sign reversed by the descent: when the entropy is below the target the loss pushes $\alpha$ up, and vice versa. The standard heuristic for the target is $\bar H = -d$, minus the dimension of the action space, and the optimisation is done on $\log \alpha$ to keep the temperature positive.

### Algorithm

The SAC algorithm is then:

1. Use a stochastic actor (squashed Gaussian), two critics with their target copies, and a learnable temperature
2. After an initial phase of random actions, act in the environment by sampling from the actor, and store the transition in the replay memory
3. Sample a mini-batch; sample next actions $a' \sim \pi_\theta(\cdot \mid s')$; compute the soft target $y = r + \gamma (1-d)\, (\min_i Q_{\phi_i^-}(s',a') - \alpha \log \pi_\theta(a' \mid s'))$ and update both critics
4. Sample reparameterised actions $a = f_\theta(s,\xi)$ for the states of the mini-batch and update the actor by gradient descent on $\alpha \log \pi_\theta(a \mid s) - \min_i Q_{\phi_i}(s,a)$
5. Update the temperature by gradient descent on $-\alpha(\log \pi_\theta(a \mid s) + \bar H)$
6. Softly update the target critics

The actor module outputs the mean and the log-standard-deviation (clamped to a sensible range for numerical stability), and provides a `sample` method that returns a reparameterised action together with its log-probability, including the $\tanh$ correction, and a `deterministic` method, which returns the squashed mean, for evaluation:

In [ ]:
class GaussianActor(torch.nn.Module):

    def __init__(self, env, first_hidden_layer=256, second_hidden_layer=256, log_std_min=-20.0, log_std_max=2.0):
        super().__init__()

        state_size = env.observation_space.shape[0]
        action_size = env.action_space.shape[0]

        self.action_max = float(env.action_space.high[0])
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max

        # shared trunk
        self.network = torch.nn.Sequential(
            torch.nn.Linear(state_size, first_hidden_layer),
            torch.nn.ReLU(),
            torch.nn.Linear(first_hidden_layer, second_hidden_layer),
            torch.nn.ReLU(),
        )

        # two heads: the mean and the log standard deviation of the Gaussian
        self.mean_head = torch.nn.Linear(second_hidden_layer, action_size)
        self.log_std_head = torch.nn.Linear(second_hidden_layer, action_size)

    def forward(self, state):
        features = self.network(state)
        mean = self.mean_head(features)
        log_std = self.log_std_head(features).clamp(self.log_std_min, self.log_std_max)
        return mean, log_std

    def sample(self, state):

        mean, log_std = self.forward(state)
        std = log_std.exp()

        # reparameterisation: u = mean + std * xi, with xi ~ N(0, I)
        normal = torch.distributions.Normal(mean, std)
        u = normal.rsample()

        # squash into the action bounds
        squashed = torch.tanh(u)
        action = self.action_max * squashed

        # log-probability with the change-of-variables correction for the tanh and the scaling
        log_prob = normal.log_prob(u) - torch.log(self.action_max * (1.0 - squashed.pow(2)) + 1e-6)
        log_prob = log_prob.sum(dim=-1)

        return action, log_prob

    def deterministic(self, state):
        mean, _ = self.forward(state)
        return self.action_max * torch.tanh(mean)

We create the actor, the twin critics with their targets, the temperature (as $\log \alpha$, so that $\alpha$ stays positive) and the optimizers. The target entropy is minus the dimension of the action space:

In [ ]:
actor = GaussianActor(pendulum)
critic_1 = Critic(pendulum)
critic_2 = Critic(pendulum)
target_critic_1 = copy.deepcopy(critic_1)
target_critic_2 = copy.deepcopy(critic_2)

log_alpha = torch.zeros(1, requires_grad=True)
target_entropy = -float(action_size)

actor_optimizer = torch.optim.Adam(actor.parameters(), lr=actor_learning_rate)
critic_optimizer = torch.optim.Adam(list(critic_1.parameters()) + list(critic_2.parameters()), lr=critic_learning_rate)
alpha_optimizer = torch.optim.Adam([log_alpha], lr=actor_learning_rate)

The exploratory policy is the actor itself, sampling; the deterministic policy, for evaluation, uses the mean:

In [ ]:
def sac_sample_pi(state):

    # sample an action from the actor (exploration is built in)
    with torch.no_grad():
        action, _ = actor.sample(torch.as_tensor(state, dtype=torch.float32))

    return action.numpy()


def sac_pi(state):

    # the deterministic action: the squashed mean
    with torch.no_grad():
        action = actor.deterministic(torch.as_tensor(state, dtype=torch.float32))

    return action.numpy()

The optimization step implements the three updates, critics, actor and temperature, followed by the soft update of the target critics:

In [ ]:
def optimize_sac(gamma=0.99, batch_size=256, tau=0.005):

    states, actions, rewards, next_states, failures = sample_experiences(batch_size)
    alpha = log_alpha.exp().detach()

    # soft target: next actions sampled from the current actor, entropy term included
    with torch.no_grad():
        next_actions, next_log_probs = actor.sample(next_states)
        next_q = torch.min(target_critic_1(next_states, next_actions), target_critic_2(next_states, next_actions))
        target = rewards + gamma * (1.0 - failures) * (next_q - alpha * next_log_probs)

    # both critics regress towards the soft target
    critic_loss = (target - critic_1(states, actions)).pow(2).mean() + (target - critic_2(states, actions)).pow(2).mean()

    critic_optimizer.zero_grad()
    critic_loss.backward()
    critic_optimizer.step()

    # actor: reparameterised actions, pushed towards high soft value
    new_actions, log_probs = actor.sample(states)
    new_q = torch.min(critic_1(states, new_actions), critic_2(states, new_actions))
    actor_loss = (alpha * log_probs - new_q).mean()

    actor_optimizer.zero_grad()
    actor_loss.backward()
    actor_optimizer.step()

    # temperature: keep the entropy close to the target
    alpha_loss = -(log_alpha.exp() * (log_probs.detach() + target_entropy)).mean()

    alpha_optimizer.zero_grad()
    alpha_loss.backward()
    alpha_optimizer.step()

    # soft updates of the target critics
    soft_update(target_critic_1, critic_1, tau)
    soft_update(target_critic_2, critic_2, tau)

    return critic_loss.item(), actor_loss.item(), alpha.item()

The training loop is that of DDPG with the sampling actor in place of the noisy one:

In [ ]:
def sac(env, max_steps, eval_every=1000, start_steps=1000, gamma=0.99, batch_size=256, tau=0.005):

    # a separate, seeded environment for evaluation
    eval_env = gymnasium.make(env.spec.id)
    eval_env.reset(seed=int(env.np_random.integers(1_000_000)))

    scores = []
    state, _ = env.reset()

    for step in range(1, max_steps + 1):

        # random actions at the beginning, then the stochastic actor
        if step <= start_steps:
            action = env.action_space.sample()
        else:
            action = sac_sample_pi(state)

        # interact with the environment and store the transition
        next_state, reward, terminated, truncated, _ = env.step(action)
        store_experience((state, action, reward, next_state, terminated))

        # move on; reset the environment at the end of the episode
        state = next_state
        if terminated or truncated:
            state, _ = env.reset()

        # one optimization step per environment step, once the memory has enough transitions
        if step > start_steps:
            optimize_sac(gamma, batch_size, tau)

        # evaluate the deterministic policy at regular intervals
        if step % eval_every == 0:
            score = evaluate(eval_env, sac_pi, episodes=5)
            scores.append(score)
            print(f"Step {step:06}, score {score:07.1f}", end="\r", flush=True)

    return scores

### Solve Pendulum

We run the experiment and compare SAC with TD3:

In [ ]:
def setup_sac():
    global actor, critic_1, critic_2, target_critic_1, target_critic_2, log_alpha, actor_optimizer, critic_optimizer, alpha_optimizer, replay_memory

    actor = GaussianActor(pendulum, first_hidden_layer, second_hidden_layer)
    critic_1 = Critic(pendulum, first_hidden_layer, second_hidden_layer)
    critic_2 = Critic(pendulum, first_hidden_layer, second_hidden_layer)
    target_critic_1 = copy.deepcopy(critic_1)
    target_critic_2 = copy.deepcopy(critic_2)

    log_alpha = torch.zeros(1, requires_grad=True)

    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=actor_learning_rate)
    critic_optimizer = torch.optim.Adam(list(critic_1.parameters()) + list(critic_2.parameters()), lr=critic_learning_rate)
    alpha_optimizer = torch.optim.Adam([log_alpha], lr=actor_learning_rate)

    replay_memory = {
        "size": memory_size,
        "buffer": np.empty(memory_size, dtype=experience_type),
        "index": 0,
        "entries": 0,
    }

def train_sac(env, max_steps, eval_every):
    return sac(env, max_steps, eval_every, start_steps, gamma, batch_size, tau)

# run the experiment
sac_results = experiment(
    env=pendulum,
    train=train_sac,
    setup=setup_sac,
    max_steps=30_000,
    name="sac",
)

In [ ]:
plot_comparison(
    td3_results,
    sac_results,
    first_label="TD3",
    second_label="SAC",
    first_color="blue",
    second_color="green",
)

On Pendulum, SAC and TD3 behave similarly: both reach a good controller quickly and keep it, and the differences are within the evaluation noise. What SAC gives on this simple problem is mainly **robustness to hyperparameters**: there is no exploration noise to tune, and the temperature adapts the amount of randomness to the environment and to the stage of training, typically starting high, when the critic knows little, and decreasing as the policy sharpens. On the larger continuous control benchmarks this adaptive exploration pays off, and SAC is generally the more reliable of the two; it also tends to be more robust to the scale of the rewards, thanks to the automatic temperature. For these reasons it is today the default choice for off-policy continuous control.

## Solve Lunar Lander (Continuous)

Lunar Lander has a continuous version, in which the two actions are the throttle of the main engine and that of the lateral engines, both in $[-1, 1]$ (the main engine fires only when its throttle is positive, the lateral engines only beyond a dead zone around zero). The state, the rewards and the solved threshold of $200$ are those of the discrete version used in Chapters 08 and 09, where the on-policy agents, after hundreds of thousands of steps, had learned to hover and drift towards the pad but not to land reliably. We apply SAC with the same hyperparameters, a larger random warm-up and a longer training budget, and we evaluate less frequently:

In [ ]:
lunar_lander = gymnasium.make('LunarLanderContinuous-v3', render_mode="rgb_array")

print('State Space: ', lunar_lander.observation_space)
print('Action Space: ', lunar_lander.action_space)

In [ ]:
# set the seeds for reproducibility of results
seed = 34

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

lunar_lander.reset(seed=seed)
lunar_lander.action_space.seed(seed)

# sizes of the new environment, used by the replay memory and the policies
state_size = lunar_lander.observation_space.shape[0]
action_size = lunar_lander.action_space.shape[0]
action_max = float(lunar_lander.action_space.high[0])

experience_type = np.dtype([
    ('state',      np.float32, state_size),
    ('action',     np.float32, action_size),
    ('reward',     np.float32),
    ('next_state', np.float32, state_size),
    ('failure',    np.int8)
])

# fresh networks, temperature, optimizers and replay memory for the new environment
actor = GaussianActor(lunar_lander, first_hidden_layer, second_hidden_layer)
critic_1 = Critic(lunar_lander, first_hidden_layer, second_hidden_layer)
critic_2 = Critic(lunar_lander, first_hidden_layer, second_hidden_layer)
target_critic_1 = copy.deepcopy(critic_1)
target_critic_2 = copy.deepcopy(critic_2)

log_alpha = torch.zeros(1, requires_grad=True)
target_entropy = -float(action_size)

actor_optimizer = torch.optim.Adam(actor.parameters(), lr=actor_learning_rate)
critic_optimizer = torch.optim.Adam(list(critic_1.parameters()) + list(critic_2.parameters()), lr=critic_learning_rate)
alpha_optimizer = torch.optim.Adam([log_alpha], lr=actor_learning_rate)

replay_memory = {
    "size": memory_size,
    "buffer": np.empty(memory_size, dtype=experience_type),
    "index": 0,
    "entries": 0,
}

random_score = evaluate(lunar_lander, lambda state: lunar_lander.action_space.sample(), episodes=100)
print("Random score: ", random_score)

In [ ]:
scores = sac(lunar_lander, max_steps=300_000, eval_every=10_000, start_steps=10_000)

In [ ]:
eval_every = 10_000

plt.figure(figsize=(12, 6))
plt.title("SAC on Lunar Lander (continuous)")
plt.xlabel("Environment steps")
plt.ylabel("Score")
plt.plot((np.arange(len(scores)) + 1) * eval_every, scores, color="green", linewidth=2)
plt.axhline(200, color="gray", linestyle="--", linewidth=1, label="Solved")
plt.legend()
plt.show()

In [ ]:
agent_score = evaluate(lunar_lander, sac_pi, episodes=10)

print("Random score", random_score)
print("Agent score", agent_score)

In [ ]:
show_policy(lunar_lander, sac_pi, n_episodes=1, max_steps=1000)

The contrast with the on-policy agents of Chapters 08 and 09 is striking. SAC learns to land: the score climbs steadily from the $-200$ of the random policy, crosses zero within the first hundred thousand steps and approaches or exceeds the solved threshold within the training budget, with far smaller oscillations than A2C showed on the same task. Note that the budget, three hundred thousand steps, is a fraction of what A2C consumed without solving the task. Two things make the difference. Every transition is stored and **replayed** many times, so that the agent extracts far more learning signal from the same experience; and the actor is updated along the **gradient of the critic**, a direction, rather than along a noisy return-weighted score. The price is computation: one gradient step on three networks and the temperature for every environment step, with mini-batches of $256$ transitions, so that in wall-clock time SAC is much slower per environment step than A2C, and a good GPU matters here more than anywhere else in the course.

## Limitations and Outlook

The algorithms of this chapter complete the map of deep reinforcement learning that we have been drawing since Chapter 07. Value-based methods are off-policy and sample efficient but need discrete actions; policy gradient methods handle any action space but are on-policy and sample hungry; the deterministic policy gradient, and its stochastic descendant SAC, take the best of both: an actor for continuous actions and a critic trained off-policy from replayed experience. DDPG, TD3 and SAC share this structure, and the progression between them is a progression in **controlling the errors of the critic**: pessimistic targets, smoothed targets, delayed actor updates, stochastic policies with adaptive entropy. Three limitations remain.

First, these methods inherit the **deadly triad** of Chapter 07 in full: function approximation, bootstrapping and off-policy data. The remedies of TD3 and SAC make divergence rare, but they do not make it impossible, and the hyperparameters, learning rates above all, still matter. Second, the actor **exploits the critic**: wherever the critic is wrong, the actor will find it, and the data needed to correct the error may never be collected if the wrong action is never actually good. This is a mild problem when the agent collects its own data, because it eventually visits the actions it believes in and the critic is corrected; it becomes severe when the data are fixed in advance and the agent cannot interact, the **offline** setting of Chapter 12. Third, off-policy actor-critics are computationally **expensive** per environment step and, SAC less than the others, sensitive to the scale of the rewards, and in settings where environment interaction is cheap, such as simulated robotics and games, the on-policy alternative is often preferred for its simplicity and robustness. Making the on-policy approach competitive in sample efficiency is the subject of the next chapter.